# Entraînement MKAN — Pipeline complet

**Flux** :
1. Chargement de trois simulations indépendantes (seeds distincts) générées par `generate_sim_dataset.py` via `MOMTSIM` :
   - **Train** — `MOMTSIM/config/featuresLog.parquet` (500 K clients, seed 1000, ~5,5 M tx)
   - **Val**   — `data/val_features.parquet`  (150 K clients, seed 1001, ~1,65 M tx)
   - **Test**  — `data/test_features.parquet` (150 K clients, seed 1002, ~1,65 M tx)
2. Prétraitement uniforme : remplissage NaN → transforms log (shifts ajustés sur le train) → normalisation z-score (μ, σ ajustés sur le train)
3. Construction des fenêtres glissantes W × 12 features par compte `nameOrig` (eq. 4.16)
4. Entraînement `MKANScorer` via `mkan_total_loss` (section 4.3.4)
5. Évaluation MCC / AUC-ROC (baseline XGBoost = 0,82)
6. Détection de dérive JS + extension de grille (section 4.4)
7. Élagage + régression symbolique → rapport d'audit COBAC (section 4.4.7)

In [1]:
import sys, os

# Le notebook est dans MKAN/ → le parent (..) est Modelisation/
ROOT        = os.path.abspath("..")
MOMTSIM_DIR = os.path.join(ROOT, "MOMTSIM")
for p in [ROOT, MOMTSIM_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Package MKAN (MKAN/__init__.py)
from MKAN import (
    MKANScorer, mkan_total_loss,
    js_divergence, detect_drift_region,
    extract_full_model_report,
)

# Détection GPU : DirectML (Intel Iris Xe) → XPU (Intel natif) → CUDA → CPU
try:
    import torch_directml
    DEVICE = torch_directml.device()
    print(f"Device : Intel Iris Xe via DirectML  →  {DEVICE}")
except ImportError:
    try:
        if torch.xpu.is_available():
            DEVICE = torch.device("xpu")
            print(f"Device : Intel XPU (natif PyTorch)  →  {DEVICE}")
        else:
            raise AttributeError
    except AttributeError:
        if torch.cuda.is_available():
            DEVICE = torch.device("cuda")
            print(f"Device : CUDA  {torch.cuda.get_device_name(0)}")
        else:
            DEVICE = torch.device("cpu")
            print("Device : CPU  (installer torch-directml pour activer l'Intel Iris Xe)")

print("PyTorch", torch.__version__, "| CUDA", torch.cuda.is_available())
print("sys.path[0:2] :", sys.path[:2])

Device : Intel Iris Xe via DirectML  →  privateuseone:0
PyTorch 2.4.1+cpu | CUDA False
sys.path[0:2] : ['m:\\Ecole\\Mémoire\\Modelisation\\MOMTSIM', 'm:\\Ecole\\Mémoire\\Modelisation']


## 1. Configuration

In [2]:
# ── Chemins ────────────────────────────────────────────────────────────────
FEATURES_FILE = os.path.join("..", "MOMTSIM", "config", "featuresLog.parquet")  # train (5.5 M tx)
VAL_FILE      = os.path.join("..", "data", "val_features.parquet")               # sim indépendante seed=1001
TEST_FILE     = os.path.join("..", "data", "test_features.parquet")              # sim indépendante seed=1002
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Features (eq. 3.8–3.19, section 3.2.6) ────────────────────────────────
FEATURE_COLS = [
    "delta_B_orig",          # eq. 3.8   ΔB_orig
    "delta_B_dest",          # eq. 3.9   ΔB_dest
    "r1",                    # eq. 3.10  montant / ancien solde orig
    "r2",                    # eq. 3.11  montant / nouveau solde orig
    "flag_anomalie",         # eq. 3.12  Flag_anomalie ∈ {0,1}
    "delta_commission",      # eq. 3.13  δ_commission (smurfing)
    "var_agent_split",       # eq. 3.14  Var_agent (split deposit)
    "rho_rupture",           # eq. 3.15  ρ_rupture (fake credentials)
    "rho_refund",            # eq. 3.16  ρ_refund (refund fraud)
    "v1h",                   # eq. 3.17  V_1h (vélocité)
    "flag_nuit",             # eq. 3.18  Flag_nuit ∈ {0,1}
    "rho_nouveau",           # eq. 3.19  ρ_nouveau (destinataires inconnus)
]
TARGET_COL   = "isFraud"
ACCOUNT_COL  = "nameOrig"
TIME_COL     = "step"

# ── Architecture MKAN (section 4.2) ───────────────────────────────────────
INPUT_SIZE  = len(FEATURE_COLS)   # 12
HIDDEN_SIZE = 32                  # dimension h_t / c_t
W           = 10                  # fenêtre glissante (pas de temps)
M_GAUSS     = 8                   # centres gaussiens par arête (section 4.2.2)
K_FOURIER   = 2                   # harmoniques Fourier    (section 4.2.2)

# ── Entraînement (section 4.3.4) ──────────────────────────────────────────
BATCH_SIZE  = 256
LR          = 1e-3
N_EPOCHS    = 40
LAM         = 1e-2    # force de régularisation globale
MU1         = 1.0     # poids norme L1
MU2         = 0.5     # poids entropie

# ── Découpage temporel ─────────────────────────────────────────────────────
TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.15   # (test = 1 - 0.70 - 0.15 = 0.15)

# ── Seuil de drift JS (section 4.4.2, eq. 4.19) ───────────────────────────
JS_THRESHOLD = 0.05

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# DEVICE est défini dans la cellule d'imports (détection cascadée DirectML → XPU → CUDA → CPU)
print(f"Dispositif : {DEVICE}")
print(f"VAL_FILE  : {os.path.abspath(VAL_FILE)}")
print(f"TEST_FILE : {os.path.abspath(TEST_FILE)}")

Dispositif : privateuseone:0
VAL_FILE  : m:\Ecole\Mémoire\Modelisation\data\val_features.parquet
TEST_FILE : m:\Ecole\Mémoire\Modelisation\data\test_features.parquet


## 2. Chargement et exploration des données

In [3]:
df = pd.read_parquet(FEATURES_FILE)

# Remplissage des NaN avant toute normalisation
# delta_commission et var_agent_split sont NaN pour les tx où la feature ne s'applique pas :
# → 0 est la valeur neutre sémantiquement correcte (aucune activité de mule / aucun split)
nan_counts = df[FEATURE_COLS].isna().sum()
if nan_counts.any():
    print("NaN détectés (remplacement par 0) :")
    print(nan_counts[nan_counts > 0])
    df[FEATURE_COLS] = df[FEATURE_COLS].fillna(0.0)

print(f"\nTransactions totales : {len(df):,}")
print(f"Taux de fraude global : {df[TARGET_COL].mean():.3f}")
print(f"Steps : {df[TIME_COL].min()} → {df[TIME_COL].max()}")
print(f"Comptes uniques (nameOrig) : {df[ACCOUNT_COL].nunique():,}")
if df['fraudScenario'].notna().any():
    print("\nRépartition par scénario :")
    print(df.loc[df[TARGET_COL], 'fraudScenario'].value_counts())

NaN détectés (remplacement par 0) :
delta_commission    5569643
var_agent_split     4027625
dtype: int64

Transactions totales : 5,569,643
Taux de fraude global : 0.236
Steps : 0 → 1439
Comptes uniques (nameOrig) : 510,288

Répartition par scénario :
fraudScenario
ATO          711729
SPLIT_DEP    311292
SMURFING     220387
REFUND        72000
FAKE_CRED       216
Name: count, dtype: int64


In [4]:
# Statistiques des 12 features
df[FEATURE_COLS + [TARGET_COL]].describe().round(4)

,delta_B_orig,delta_B_dest,r1,r2,delta_commission,var_agent_split,rho_rupture,rho_refund,v1h,rho_nouveau
count,5.569643e+06,5569643.0,5.569643e+06,5.569643e+06,5569643.0,5.569643e+06,5.569643e+06,5.569643e+06,5.569643e+06,5.569643e+06
mean,-3.982394e+06,-5202791.5,1.018968e+10,8.503140e+09,0.0,7.447831e+06,1.059930e+12,9.610275e+04,4.482700e+00,4.168000e-01
std,1.301349e+07,14867041.0,5.965212e+11,2.834362e+11,0.0,2.463808e+08,6.011208e+12,5.169761e+05,2.935790e+01,3.062000e-01
min,-1.386788e+08,-188083152.0,-5.757385e+07,-1.016825e+05,0.0,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00
25%,-2.100960e+05,-563142.0,2.000000e-04,2.000000e-04,0.0,0.000000e+00,1.550000e-02,0.000000e+00,1.000000e+00,2.000000e-01
50%,2.860000e+03,1912.0,8.600000e-03,1.230000e-02,0.0,0.000000e+00,3.930000e-01,0.000000e+00,1.000000e+00,3.333000e-01
75%,1.318825e+05,136876.0,3.807000e-01,5.556000e-01,0.0,0.000000e+00,3.319500e+00,0.000000e+00,1.000000e+00,5.833000e-01
max,1.148283e+08,114823920.0,1.132354e+14,3.274633e+13,0.0,4.973312e+10,1.262110e+14,3.000000e+06,3.410000e+02,1.000000e+00


## 2.5 Détection et transformation log — même protocole que MOMTSIM (§4.1, éq. 4.5)

`TopologyValidator` (MOMTSIM/src/viz.py) calcule $D_{KS}$ comme la distance maximale entre
la ECDF normalisée et $\Phi$ (éq. 4.5). Seuil : $D_{KS} \geq 0.15$ → transformation.

Transformation appliquée (identique à `apply_recommended_transforms()`) :
$$x' = \log\!\left(1 + \max(x - x_{\min},\, 0)\right)$$
i.e. décalage vers 0 si la feature est négative, puis $\log(1+\cdot)$.

In [5]:
import sys as _sys, os as _os
_momtsim_src = _os.path.join(_os.path.abspath(".."), "MOMTSIM", "src")
if _momtsim_src not in _sys.path:
    _sys.path.insert(0, _momtsim_src)

from viz import TopologyValidator, BINARY_FEATURES

# ── Validation topologique sur le DataFrame complet (avant split) ──────────────
_validator = TopologyValidator(df, features=FEATURE_COLS)
_validator.normalize()           # éq. 4.1  z-score interne au validateur
ks_results = _validator.ks_per_feature()   # éq. 4.5  ECDF vs Φ, sample=5000

print(f"Test KS vs loi normale (éq. 4.5)    seuil D_KS ≥ 0.15\n")
print(f"  {'feature':25s}  {'D_KS':>6}  décision")
print(f"  {'-'*56}")
for col in FEATURE_COLS:
    if col in BINARY_FEATURES:
        print(f"  {col:25s}  {'':>6}  ignoré (binaire)")
        continue
    ks = ks_results.get(col)
    flag = "✓ TRANSFORM" if ks is not None and ks >= 0.15 else " OK"
    ks_str = f"{ks:.3f}" if ks is not None else "?"
    print(f"  {col:25s}  {ks_str:>6}  {flag}")

LOG_COLS = _validator.report.get("features_needing_transform", [])
print(f"\n→ {len(LOG_COLS)} features à transformer : {LOG_COLS}")

# LOG_SHIFTS : min de chaque feature sur le train, pour appliquer le même décalage à val/test
LOG_SHIFTS: dict[str, float] = {}
for col in LOG_COLS:
    x = df[col].values
    col_min = float(x.min())
    LOG_SHIFTS[col] = col_min
    shifted = x - col_min if col_min < 0 else x
    df[col] = np.log1p(shifted)

if LOG_COLS:
    print("\nStatistiques post-transformation :")
    print(df[LOG_COLS].describe().round(4).to_string())

Test KS vs loi normale (éq. 4.5)    seuil D_KS ≥ 0.15

  feature                      D_KS  décision
  --------------------------------------------------------
  delta_B_orig                0.395  ✓ TRANSFORM
  delta_B_dest                0.374  ✓ TRANSFORM
  r1                          0.505  ✓ TRANSFORM
  r2                          0.511  ✓ TRANSFORM
  flag_anomalie                      ignoré (binaire)
  delta_commission            0.500  ✓ TRANSFORM
  var_agent_split             0.491  ✓ TRANSFORM
  rho_rupture                 0.504  ✓ TRANSFORM
  rho_refund                  0.535  ✓ TRANSFORM
  v1h                         0.474  ✓ TRANSFORM
  flag_nuit                          ignoré (binaire)
  rho_nouveau                 0.128   OK

→ 9 features à transformer : ['delta_B_orig', 'delta_B_dest', 'r1', 'r2', 'delta_commission', 'var_agent_split', 'rho_rupture', 'rho_refund', 'v1h']

Statistiques post-transformation :
       delta_B_orig  delta_B_dest            r1            r2  d

## 3. Découpage train / val / test + Normalisation (section 4.1.1, eq. 4.1)

Trois simulations indépendantes (seeds distincts) pour éliminer tout biais de déplétion ATO :

| Split | Fichier source | Clients | Steps | Seed | ~Transactions |
|-------|---------------|---------|-------|------|---------------|
| **Train** | `MOMTSIM/config/featuresLog.parquet` | 500 000 | 1 440 | 1000 | 5,5 M |
| **Val**   | `data/val_features.parquet`          | 150 000 | 1 440 | 1001 | ~1,65 M |
| **Test**  | `data/test_features.parquet`         | 150 000 | 1 440 | 1002 | ~1,65 M |

Les fichiers val/test sont générés par `generate_sim_dataset.py` (à exécuter une seule fois).
La normalisation $\tilde{x}_{ij} = (x_{ij} - \mu_j) / (\sigma_j + \varepsilon)$ est ajustée **uniquement sur le train**.

In [6]:
def _apply_log_transforms(d: pd.DataFrame) -> None:
    """Applique les mêmes transforms log que sur le train (shifts fixés sur LOG_SHIFTS)."""
    d[FEATURE_COLS] = d[FEATURE_COLS].fillna(0.0)
    for col in LOG_COLS:
        x = d[col].values
        train_min = LOG_SHIFTS[col]
        shifted = np.maximum(x - train_min, 0) if train_min < 0 else np.maximum(x, 0)
        d[col] = np.log1p(shifted)


for path, name in [(VAL_FILE, "val"), (TEST_FILE, "test")]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Fichier {name} introuvable : {os.path.abspath(path)}\n"
            "→ Générer les données avec :\n"
            "    python generate_sim_dataset.py sim_config_val.json\n"
            "    python generate_sim_dataset.py sim_config_test.json"
        )

df_train = df.copy()   # df a déjà les log-transforms appliquées (cellule précédente)

df_val  = pd.read_parquet(VAL_FILE)
df_test = pd.read_parquet(TEST_FILE)

_apply_log_transforms(df_val)
_apply_log_transforms(df_test)

print(f"Train : {len(df_train):,} tx  ({df_train[TARGET_COL].mean():.3f} fraude)"
      f"   {df_train[ACCOUNT_COL].nunique():,} comptes")
print(f"Val   : {len(df_val):,} tx  ({df_val[TARGET_COL].mean():.3f} fraude)"
      f"   {df_val[ACCOUNT_COL].nunique():,} comptes")
print(f"Test  : {len(df_test):,} tx  ({df_test[TARGET_COL].mean():.3f} fraude)"
      f"   {df_test[ACCOUNT_COL].nunique():,} comptes")

# Normalisation fit sur le train uniquement (évite la fuite de données)
mu_train  = df_train[FEATURE_COLS].mean()
std_train = df_train[FEATURE_COLS].std().clip(lower=1e-6)

for d in [df_train, df_val, df_test]:
    d[FEATURE_COLS] = (d[FEATURE_COLS] - mu_train) / std_train

print("\nNormalisation appliquée.")

Train : 5,569,643 tx  (0.236 fraude)   510,288 comptes
Val   : 1,578,746 tx  (0.191 fraude)   153,159 comptes
Test  : 1,579,438 tx  (0.191 fraude)   153,060 comptes

Normalisation appliquée.


## 4. Construction des fenêtres glissantes (eq. 4.16)

Pour chaque compte `nameOrig`, on crée toutes les fenêtres de $W$ transactions consécutives.
Le label d'une fenêtre est l'étiquette `isFraud` de la **dernière** transaction de la fenêtre.

In [7]:
from tqdm.auto import tqdm as _tqdm

def make_windows(df_split: pd.DataFrame, W: int, desc: str = "Fenêtres", leave: bool = True):
    """
    Fenêtres glissantes par compte (eq. 4.16) : (batch, W, 12) → label de la dernière tx.
    Comptes avec < W transactions sont ignorés (fenêtre impossible).
    leave=False supprime la barre après complétion (utile depuis la recherche heuristique).
    """
    X_list, y_list = [], []
    feat = df_split[FEATURE_COLS].values.astype(np.float32)
    targ = df_split[TARGET_COL].values.astype(np.float32)
    acct = df_split[ACCOUNT_COL].values
    step = df_split[TIME_COL].values

    _, unique_starts = np.unique(acct, return_index=True)
    unique_ends = np.append(unique_starts[1:], len(acct))

    for start, end in _tqdm(zip(unique_starts, unique_ends),
                             total=len(unique_starts),
                             desc=desc, unit="compte", leave=leave):
        n = end - start
        if n < W:
            continue
        order = np.argsort(step[start:end])
        Xacc = feat[start:end][order]
        yacc = targ[start:end][order]
        for i in range(n - W + 1):
            X_list.append(Xacc[i:i+W])
            y_list.append(yacc[i+W-1])

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y


for d in [df_train, df_val, df_test]:
    d.sort_values([ACCOUNT_COL, TIME_COL], inplace=True)
    d.reset_index(drop=True, inplace=True)

X_train, y_train = make_windows(df_train, W, desc="Train")
X_val,   y_val   = make_windows(df_val,   W, desc="Val  ")
X_test,  y_test  = make_windows(df_test,  W, desc="Test ")

print(f"\nFenêtres train : {len(X_train):,}  (fraude : {y_train.mean():.3f})")
print(f"Fenêtres val   : {len(X_val):,}  (fraude : {y_val.mean():.3f})")
print(f"Fenêtres test  : {len(X_test):,}  (fraude : {y_test.mean():.3f})")

Train:   0%|          | 0/510288 [00:00<?, ?compte/s]

Val  :   0%|          | 0/153159 [00:00<?, ?compte/s]

Test :   0%|          | 0/153060 [00:00<?, ?compte/s]


Fenêtres train : 1,749,109  (fraude : 0.439)
Fenêtres val   : 450,955  (fraude : 0.346)
Fenêtres test  : 452,349  (fraude : 0.346)


In [8]:
def to_loader(X, y, batch_size, shuffle):
    X_t = torch.from_numpy(np.ascontiguousarray(X, dtype=np.float32))
    y_t = torch.from_numpy(np.ascontiguousarray(y, dtype=np.float32))
    ds  = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

train_loader = to_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = to_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)
test_loader  = to_loader(X_test,  y_test,  BATCH_SIZE, shuffle=False)

## 5. Optimisation des hyperparamètres (recherche heuristique)

Algorithme génétique amélioré (Hao & Solnon, 2024 ; Yang Lu & Felix Zhan, 2024) :

- **Mutation adaptative** : taux de mutation décroissant (refroidissement, analogie recuit simulé)
- **Contrôle de la diversité** : injection d'individus aléatoires si la population converge prématurément
- **Composante EDA** : 1/3 des enfants générés selon un modèle probabiliste sur les meilleurs individus

**Hyperparamètres explorés** : `hidden_size`, `M`, `K`, `lam`, `mu1`, `mu2`, `lr`

> Chaque évaluation entraîne un `MKANScorer` pendant `N_EPOCHS_SEARCH` epochs et retourne le MCC sur `val_loader`. Les meilleurs hyperparamètres (`best_hp`) sont utilisés pour l'entraînement final (section 13).

In [9]:
import importlib.util as _ilu, sys as _sys, os as _os

# Rechargement complet du package MKAN pour prendre en compte toute modification des .py
for _k in list(_sys.modules.keys()):
    if _k.startswith("MKAN"):
        del _sys.modules[_k]

from MKAN import (
    MKANScorer, mkan_total_loss,
    js_divergence, detect_drift_region,
    extract_full_model_report,
)

# Chargement du module heuristic_search via importlib (rechargé à chaque exécution)
_spec = _ilu.spec_from_file_location(
    "heuristic_search",
    _os.path.join(_os.path.abspath("."), "heuristic_search.py"),
)
_mod = _ilu.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

build_mkan_fitness   = _mod.build_mkan_fitness
RechercheHeuristique = _mod.RechercheHeuristique
ESPACE_MKAN_DEFAULT  = _mod.ESPACE_MKAN_DEFAULT
ESPACE_MKAN_ETENDU   = _mod.ESPACE_MKAN_ETENDU

print("MKAN rechargé + RechercheHeuristique chargée.")
print(f"  ESPACE_MKAN_DEFAULT  ({len(ESPACE_MKAN_DEFAULT)} HP) : {list(ESPACE_MKAN_DEFAULT.keys())}")
print(f"  ESPACE_MKAN_ETENDU   ({len(ESPACE_MKAN_ETENDU)} HP) : {list(ESPACE_MKAN_ETENDU.keys())}")

MKAN rechargé + RechercheHeuristique chargée.
  ESPACE_MKAN_DEFAULT  (7 HP) : ['hidden_size', 'M', 'K', 'lam', 'mu1', 'mu2', 'lr']
  ESPACE_MKAN_ETENDU   (9 HP) : ['hidden_size', 'M', 'K', 'lam', 'mu1', 'mu2', 'lr', 'W', 'batch_size']


In [10]:
# Choix de l espace de recherche
# ESPACE_MKAN_DEFAULT  ->  7 HP (hidden_size, M, K, lam, mu1, mu2, lr)
#                          W et batch_size fixes (W=10, batch=512)
# ESPACE_MKAN_ETENDU   ->  9 HP (+ W et batch_size)
#                          Reconstruction des fenetres a chaque evaluation
#                          Option B memoire : W selectionne empiriquement (section 4.2.4)

ESPACE_CHOISI = ESPACE_MKAN_ETENDU   # W in [10,15,20]

# Fonction fitness
N_EPOCHS_SEARCH = 5

MAX_TRAIN_SAMPLES = 15_000
MAX_VAL_SAMPLES   = 5_000

fitness_fn = build_mkan_fitness(
    df_train           = df_train,
    df_val             = df_val,
    make_windows       = make_windows,
    device             = DEVICE,
    input_size         = INPUT_SIZE,
    n_epochs           = N_EPOCHS_SEARCH,
    default_W          = W,
    default_batch_size = BATCH_SIZE,
    max_train_samples  = MAX_TRAIN_SAMPLES,
    max_val_samples    = MAX_VAL_SAMPLES,
    checkpoint_dir     = CHECKPOINT_DIR,
)

# n_generations=12 : budget reduit pour Iris Xe (5 epochs x 12 ind x 12 gen)
search = RechercheHeuristique(
    espace            = ESPACE_CHOISI,
    fitness_fn        = fitness_fn,
    n_generations     = 12,
    taille_population = 12,
    elite_size        = 2,
    tournament_size   = 3,
    mutation_rate     = 0.35,
    refroidissement   = 0.97,
    min_diversite     = 0.40,
    patience          = 5,
    maximize          = True,
)

budget = search.n_generations * search.taille_pop
print(f"Espace : {list(ESPACE_CHOISI.keys())}")
print(f"Budget : {budget} evaluations max  ({N_EPOCHS_SEARCH} epochs chacune)")
print(f"Sous-echantillonnage : train={MAX_TRAIN_SAMPLES:,}  val={MAX_VAL_SAMPLES:,}")

best_hp = search.fit()

print(f"{'='*55}")
print("Meilleurs hyperparametres trouves :")
for k, v in best_hp["params"].items():
    print(f"  {k:12s} = {v}")
print(f"MCC val = {best_hp['score']:.4f}")

Espace : ['hidden_size', 'M', 'K', 'lam', 'mu1', 'mu2', 'lr', 'W', 'batch_size']
Budget : 144 evaluations max  (5 epochs chacune)
Sous-echantillonnage : train=15,000  val=5,000


Générations:   0%|          | 0/12 [00:00<?, ?gén/s]

  ├─ individus:   0%|          | 0/12 [00:00<?, ?ind/s]

  Mise en cache des fenêtres W=20 (train + val)…


Fenêtres:   0%|          | 0/510288 [00:00<?, ?compte/s]

...

Gén 10/12 | best=0.9589 | moy=0.6997 | div=0.83 | μ=0.266 | =


  ├─ individus:   0%|          | 0/10 [00:00<?, ?ind/s]

Gén 11/12 | best=0.9658 | moy=0.7832 | div=0.92 | μ=0.258 | ↑ AMÉLIORATION


  ├─ individus:   0%|          | 0/10 [00:00<?, ?ind/s]

Gén 12/12 | best=0.9658 | moy=0.7157 | div=0.92 | μ=0.250 | =
Meilleurs hyperparametres trouves :
  hidden_size  = 16
  M            = 8
  K            = 2
  lam          = 0.001
  mu1          = 1.0
  mu2          = 0.5
  lr           = 0.003
  W            = 10
  batch_size   = 64
MCC val = 0.9658


In [11]:
# Résumé tabulaire de la progression
cols_resume  = ['generation', 'score', 'diversite', 'mutation_rate',
                'hidden_size', 'M', 'K', 'lam', 'lr']
cols_present = [c for c in cols_resume if c in search.resume().columns]
print("Progression par génération :")
print(search.resume()[cols_present].to_string(index=False))

# Visualisations Plotly de la recherche génétique
search.plot_convergence().show()
search.plot_parameter_space().show()
search.plot_diversity().show()

Progression par génération :
 generation    score  diversite  mutation_rate  hidden_size  M  K   lam    lr
          0 0.615347   1.000000       0.350000           64  4  4 0.001 0.003
          1 0.756625   1.000000       0.339500           16 16  4 0.001 0.003
          2 0.886603   1.000000       0.329315           16 16  2 0.010 0.003
          3 0.886603   1.000000       0.319436           16 16  2 0.010 0.003
          4 0.946225   0.916667       0.309852           32 16  4 0.001 0.003
          5 0.958927   1.000000       0.300557           16 16  4 0.005 0.003
          6 0.958927   1.000000       0.291540           16 16  4 0.005 0.003
          7 0.958927   1.000000       0.282794           16 16  4 0.005 0.003
          8 0.958927   1.000000       0.274310           16 16  4 0.005 0.003
          9 0.958927   0.833333       0.266081           16 16  4 0.005 0.003
         10 0.965783   0.916667       0.258098           16  8  2 0.001 0.003
         11 0.965783   0.916667    

In [12]:
# ══════════════════════════════════════════════════════════════════════
#  BYPASS RECHERCHE  executer si le search a ete interrompu / kernel mort
#  Lit les meilleurs HP depuis CHECKPOINT_DIR/best_search_hp.json si dispo,
#  sinon utilise des valeurs par defaut raisonnables.
# ══════════════════════════════════════════════════════════════════════
import os, json as _json

_hp_path = os.path.join(CHECKPOINT_DIR, 'best_search_hp.json')

if os.path.exists(_hp_path):
    with open(_hp_path, encoding='utf-8') as _f:
        _saved = _json.load(_f)
    best_hp = {
        'score':      _saved['score'],
        'params':     _saved['params'],
        'state_dict': None,
    }
    print(f'HP charges depuis {_hp_path}')
    print(f'MCC sauvegarde : {best_hp["score"]:.4f}')
else:
    # Aucun fichier sauvegarde  valeurs par defaut de l'espace de recherche
    print(f'Fichier non trouve : {_hp_path}')
    print('Utilisation des valeurs par defaut.')
    best_hp = {
        'score': 0.0,
        'params': {
            'hidden_size': 64,
            'M':           16,
            'K':           2,
            'lam':         1e-2,
            'mu1':         1.0,
            'mu2':         0.5,
            'lr':          1e-3,
        },
        'state_dict': None,
    }

# Dummy fitness_fn pour eviter NameError dans la cellule d'instanciation
class _DummyFitness:
    def __call__(self, *a, **kw): return -1.0
    def get_meilleur(self):
        return {'score': best_hp['score'], 'params': best_hp['params'], 'state_dict': None}
fitness_fn = _DummyFitness()

print('Parametres utilises :')
for k, v in best_hp['params'].items():
    print(f'  {k:12s} = {v}')
print('Passe directement a la cellule d instanciation du modele.')


HP charges depuis checkpoints\best_search_hp.json
MCC sauvegarde : 0.9658
Parametres utilises :
  hidden_size  = 16
  M            = 8
  K            = 2
  lam          = 0.001
  mu1          = 1.0
  mu2          = 0.5
  lr           = 0.003
  W            = 10
  batch_size   = 64
Passe directement a la cellule d instanciation du modele.


In [ ]:
# Mise a jour W et batch_size depuis les meilleurs HP (ESPACE_MKAN_ETENDU)
# Obligatoire : les fenetres et loaders doivent correspondre au W optimal trouve.
W_opt          = int(best_hp["params"].get("W",          W))
BATCH_SIZE_opt = int(best_hp["params"].get("batch_size", BATCH_SIZE))

if W_opt != W or BATCH_SIZE_opt != BATCH_SIZE:
    print(f"W optimal = {W_opt}  (etait {W})   |   batch_size = {BATCH_SIZE_opt}  (etait {BATCH_SIZE})")
    W          = W_opt
    BATCH_SIZE = BATCH_SIZE_opt

    print("Reconstruction des fenetres avec W optimal...")
    X_train, y_train = make_windows(df_train, W, desc="Train (W opt)")
    X_val,   y_val   = make_windows(df_val,   W, desc="Val   (W opt)")
    X_test,  y_test  = make_windows(df_test,  W, desc="Test  (W opt)")

    train_loader = to_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
    val_loader   = to_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)
    test_loader  = to_loader(X_test,  y_test,  BATCH_SIZE, shuffle=False)

    print(f"Fenetres train : {len(X_train):,}  |  val : {len(X_val):,}  |  test : {len(X_test):,}")
else:
    print(f"W = {W} et batch_size = {BATCH_SIZE} inchanges  pas de reconstruction.")


## 5. Instanciation du modèle MKAN (section 4.2.4, eq. 4.16)

In [ ]:
bp = best_hp['params']

model = MKANScorer(
    input_size  = INPUT_SIZE,
    hidden_size = int(bp.get('hidden_size', HIDDEN_SIZE)),
    M           = int(bp.get('M',           M_GAUSS)),
    K           = int(bp.get('K',           K_FOURIER)),
    domain      = 1.0,
).to(DEVICE)

# Warm start : charge les poids du meilleur modèle trouvé pendant la recherche.
# Ce modèle a déjà été entraîné N_EPOCHS_SEARCH epochs  on repart de là
# au lieu de repartir de zéro, ce qui économise N_EPOCHS_SEARCH epochs de calcul.
_best = fitness_fn.get_meilleur()
if _best['state_dict'] is not None:
    model.load_state_dict(_best['state_dict'])
    print(f"Warm start : poids chargés depuis la recherche "
          f"(MCC val = {_best['score']:.4f} apres {N_EPOCHS_SEARCH} epochs)")
else:
    print("Initialisation aléatoire (aucun poids sauvegardé par la recherche)")

LR  = float(bp.get('lr',  LR))
LAM = float(bp.get('lam', LAM))
MU1 = float(bp.get('mu1', MU1))
MU2 = float(bp.get('mu2', MU2))

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModèle MKAN  HP optimaux :")
for k, v in bp.items():
    print(f"  {k:12s} = {v}")
print(f"\nParamètres entraînables : {n_params:,}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR)


## 6. Boucle d'entraînement (ℓ_total, section 4.3.4)

$$\ell_{total} = \ell_{pred} + \lambda\left(\mu_1 \sum_l |\Phi_l|_1 + \mu_2 \sum_l S(\Phi_l)\right)$$

In [ ]:
@torch.inference_mode()
def evaluate(model, loader):
    """
    Métriques de détection de fraude  implémentation pure numpy (section 3.4).

    MCC  (Matthews Correlation Coefficient, eq. 3.1) :
        MCC = (TP·TN - FP·FN) / √((TP+FP)(TP+FN)(TN+FP)(TN+FN))

    AUC-ROC par règle trapézoïdale (eq. 3.2) :
        AUC = ∫ TPR d(FPR)  via np.trapz sur la courbe ROC triée par score décroissant

    PR-AUC par règle trapézoïdale (eq. 3.6) :
        PR-AUC = ∫ Précision d(Rappel)  via np.trapz sur la courbe PR

    Score de Brier (eq. 3.7) :
        Brier = (1/N) Σ (ŷᵢ - yᵢ)²

    Précision = TP / (TP + FP)   (eq. 3.4)
    Rappel    = TP / (TP + FN)   (eq. 3.4)
    F1        = 2 · Précision · Rappel / (Précision + Rappel)   (eq. 3.5)
    """
    model.eval()
    all_scores, all_labels = [], []
    for X_batch, y_batch in loader:
        all_scores.append(model(X_batch.to(DEVICE)).cpu().numpy())
        all_labels.append(y_batch.numpy())

    scores = np.concatenate(all_scores)
    labels = np.concatenate(all_labels).astype(int)
    preds  = (scores >= 0.5).astype(int)

    # Éléments de la matrice de confusion
    tp = int(((preds == 1) & (labels == 1)).sum())
    tn = int(((preds == 0) & (labels == 0)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())

    # MCC (eq. 3.1)
    num = tp * tn - fp * fn
    den = np.sqrt(float((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
    mcc = float(num / (den + 1e-12))

    # AUC-ROC trapézoïdale (eq. 3.2)
    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    if n_pos > 0 and n_neg > 0:
        sidx   = np.argsort(-scores)
        labs_s = labels[sidx].astype(float)
        tpr    = np.concatenate([[0.0], np.cumsum(labs_s)       / n_pos])
        fpr    = np.concatenate([[0.0], np.cumsum(1 - labs_s)   / n_neg])
        auc    = float(abs(np.trapz(tpr, fpr)))

        # PR-AUC trapézoïdale (eq. 3.6)
        cum_pos    = np.cumsum(labs_s)
        cum_n      = np.arange(1, len(labs_s) + 1, dtype=float)
        prec_curve = np.concatenate([[1.0], cum_pos / cum_n])
        rec_curve  = np.concatenate([[0.0], cum_pos / n_pos])
        pr_auc     = float(abs(np.trapz(prec_curve, rec_curve)))
    else:
        auc    = float("nan")
        pr_auc = float("nan")

    # Précision, Rappel, F1 (eq. 3.4–3.5)
    prec = float(tp / (tp + fp + 1e-12))
    rec  = float(tp / (tp + fn + 1e-12))
    f1   = float(2 * prec * rec / (prec + rec + 1e-12))

    # Score de Brier (eq. 3.7)
    brier = float(np.mean((scores - labels.astype(float)) ** 2))

    return dict(mcc=mcc, auc=auc, pr_auc=pr_auc, brier=brier,
                precision=prec, recall=rec, f1=f1)


In [ ]:
from tqdm.auto import tqdm as _tqdm

history = {"epoch": [], "loss": [], "pred_loss": [], "reg": [],
           "l1": [], "entropy": [],
           "val_mcc": [], "val_auc": [], "val_prauc": [], "val_brier": []}

best_val_mcc = -1.0
best_epoch   = 0

pbar = _tqdm(range(1, N_EPOCHS + 1), desc="Entraînement", unit="epoch")
for epoch in pbar:
    model.train()
    epoch_loss = epoch_pred = epoch_l1 = epoch_ent = 0.0
    n_batches  = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        loss, pred_loss, reg_l1, reg_entropy = mkan_total_loss(
            model, X_batch, y_batch, lam=LAM, mu1=MU1, mu2=MU2)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item()
        epoch_pred += pred_loss.item()
        epoch_l1   += reg_l1.item()
        epoch_ent  += reg_entropy.item()
        n_batches  += 1

    avg_loss = epoch_loss / n_batches
    avg_pred = epoch_pred / n_batches
    avg_l1   = epoch_l1  / n_batches
    avg_ent  = epoch_ent / n_batches

    history["epoch"].append(epoch)
    history["loss"].append(avg_loss)
    history["pred_loss"].append(avg_pred)
    history["reg"].append(avg_loss - avg_pred)
    history["l1"].append(avg_l1)
    history["entropy"].append(avg_ent)

    val_metrics = evaluate(model, val_loader)
    history["val_mcc"].append(val_metrics["mcc"])
    history["val_auc"].append(val_metrics["auc"])
    history["val_prauc"].append(val_metrics["pr_auc"])
    history["val_brier"].append(val_metrics["brier"])

    if val_metrics["mcc"] > best_val_mcc:
        best_val_mcc = val_metrics["mcc"]
        best_epoch   = epoch
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_mkan.pt"))

    pbar.set_postfix({
        "loss":    f"{avg_loss:.4f}",
        "val_MCC": f"{val_metrics['mcc']:.4f}",
        "val_AUC": f"{val_metrics['auc']:.4f}",
        "★" if val_metrics["mcc"] == best_val_mcc else " ": "",
    })

print(f"\nMeilleur val MCC = {best_val_mcc:.4f} (epoch {best_epoch})")

## 7. Visualisation de la convergence

In [ ]:
# Reload MKAN pour inclure MKANVisualizer si le kernel avait l'ancienne version
import importlib, sys as _sys
for _k in [_k for _k in _sys.modules if _k.startswith('MKAN')]:
    del _sys.modules[_k]
from MKAN import MKANVisualizer

# Instanciation unique du visualiseur (reutilise dans toutes les cellules suivantes)
viz = MKANVisualizer(
    model         = model,
    history       = history,
    feature_names = FEATURE_COLS,
    hidden_size   = HIDDEN_SIZE,
)
print(f'MKANVisualizer pret  {len(viz.concat_labels)} entrees: {viz.concat_labels[:3]}...')

# Tableau de bord complet : perte, regularisation, metriques (section 4.4.6)
fig_dashboard = viz.plot_training_dashboard()
fig_dashboard.write_html(os.path.join(CHECKPOINT_DIR, 'dashboard.html'))
fig_dashboard.show()
print(f'Dashboard sauvegarde : {CHECKPOINT_DIR}/dashboard.html')


## 8. Évaluation finale sur le test

Charge le meilleur checkpoint (val MCC maximal).

In [ ]:
# Rechargement du meilleur checkpoint
model.load_state_dict(
    torch.load(os.path.join(CHECKPOINT_DIR, "best_mkan.pt"),
               map_location=DEVICE, weights_only=True))

test_metrics = evaluate(model, test_loader)
print("=== Résultats TEST ===")
for k, v in test_metrics.items():
    print(f"  {k:12s} : {v:.4f}")

print(f"\nBaseline XGBoost (Azamuke et al. 2025) : MCC = 0.82, AUC = 0.97")
delta_mcc = test_metrics['mcc'] - 0.82
print(f"Écart MKAN / XGBoost   : ΔMCC = {delta_mcc:+.4f}")

In [ ]:
# Collecte des prédictions sur le jeu de test
model.eval()
all_scores_t, all_labels_t = [], []
with torch.inference_mode():
    for X_batch, y_batch in test_loader:
        all_scores_t.append(model(X_batch.to(DEVICE)).cpu().numpy())
        all_labels_t.append(y_batch.numpy())

scores_t = np.concatenate(all_scores_t)
labels_t = np.concatenate(all_labels_t).astype(int)
preds_t  = (scores_t >= 0.5).astype(int)

# Matrice de confusion (pure numpy)
tn = int(((preds_t == 0) & (labels_t == 0)).sum())
fp = int(((preds_t == 1) & (labels_t == 0)).sum())
fn = int(((preds_t == 0) & (labels_t == 1)).sum())
tp = int(((preds_t == 1) & (labels_t == 1)).sum())

cm_values   = [[tn, fp], [fn, tp]]
axis_labels = ['Légitime', 'Fraude']
annot_text  = [[str(v) for v in row] for row in cm_values]

fig_cm = go.Figure(go.Heatmap(
    z=cm_values,
    x=axis_labels,
    y=axis_labels,
    colorscale='Blues',
    text=annot_text,
    texttemplate='%{text}',
    showscale=True,
))
fig_cm.update_layout(
    title='Matrice de confusion  MKAN (test)',
    xaxis_title='Prédit',
    yaxis_title='Réel',
    width=420, height=370,
)
fig_cm.write_html(os.path.join(CHECKPOINT_DIR, 'confusion_matrix.html'))
fig_cm.show()
print(f'VN={tn}  FP={fp}  FN={fn}  VP={tp}')

## 9. Détection de dérive JS (section 4.4, eq. 4.19)

Compare la distribution des scores sur le **val** (référence) avec celle sur le **test** (déploiement simulé).

$$JS(P \| Q) = \frac{1}{2}\left[KL(P \| M) + KL(Q \| M)\right], \quad M = \frac{P+Q}{2}$$

In [ ]:
# Pool de reference (val) et pool de deploiement (test)  derniere tx de chaque fenetre
X_val_t  = torch.from_numpy(np.ascontiguousarray(X_val,  dtype=np.float32)).to(DEVICE)
X_test_t = torch.from_numpy(np.ascontiguousarray(X_test, dtype=np.float32)).to(DEVICE)

pool_ref = X_val_t[:, -1, :]    # (N_val,  12)
pool_new = X_test_t[:, -1, :]   # (N_test, 12)

# Divergence JS par feature (eq. 4.19)
# js_divergence attend des histogrammes de MEME longueur  on discretise d'abord
# les echantillons bruts avec des bins communs sur [-domain, domain]
DRIFT_BINS   = 20
DRIFT_DOMAIN = 3.0    # couvre [-3sigma, +3sigma] apres normalisation
bin_edges = np.linspace(-DRIFT_DOMAIN, DRIFT_DOMAIN, DRIFT_BINS + 1)

js_scores = []
for j, fname in enumerate(FEATURE_COLS):
    ref_j = pool_ref[:, j].cpu().numpy()
    new_j = pool_new[:, j].cpu().numpy()
    hist_ref, _ = np.histogram(ref_j, bins=bin_edges)
    hist_new, _ = np.histogram(new_j, bins=bin_edges)
    js = js_divergence(hist_ref.astype(float), hist_new.astype(float))
    js_scores.append(js)
    flag = 'DERIVE' if js > JS_THRESHOLD else 'OK'
    print(f'  {fname:25s}  JS={js:.4f}  {flag}')

drift_features = [FEATURE_COLS[j] for j, js in enumerate(js_scores) if js > JS_THRESHOLD]
print(f'\nFeatures en derive (JS > {JS_THRESHOLD}) : {drift_features}')


In [ ]:
# Extension de grille (eq. 4.17–4.18) sur les features en dérive
# detect_drift_region retourne (region, js_val) où region = (xl, xr) ou None
if drift_features:
    print('Extension de grille sur les arêtes des portes MKAN...')
    gates_list    = [model.cell.forget_gate, model.cell.input_gate,
                     model.cell.candidate_gate, model.cell.output_gate]
    N_NEW_CENTERS = 4   # centres gaussiens insérés dans la région de dérive

    extended = False
    for fname in drift_features:
        feat_idx = FEATURE_COLS.index(fname)
        region, js_val = detect_drift_region(
            pool_ref[:, feat_idx].cpu().numpy(),
            pool_new[:, feat_idx].cpu().numpy(),
        )
        print(f'  {fname}: JS={js_val:.4f}, région={region}')

        if region is not None:
            for gate in gates_list:
                new_M = gate.extend_grid(region, N_NEW_CENTERS)
            extended = True
            print(f'    -> Grille étendue à M={new_M} centres')

    if extended:
        # Reconstruction de l'optimiseur après extension (nn.Parameter remplacé)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR * 0.1)
        print(f'\nOptimiseur reconstruit (lr={LR * 0.1:.1e})')
else:
    print('Aucune dérive détectée  extension de grille non nécessaire.')

## 10. Rapport d'audit COBAC — Élagage + Régression symbolique (section 4.4.7)

In [ ]:
from tqdm.auto import tqdm as _tqdm

# Construction du pool d'audit : les portes T-KAN prennent [h_{t-1}, x_t] en entrée
# concat_size = hidden_size + input_size = 32 + 12 = 44
model.eval()
N_AUDIT     = min(500, len(X_train))
audit_X     = torch.from_numpy(np.ascontiguousarray(X_train[:N_AUDIT], dtype=np.float32)).to(DEVICE)
concat_list = []

batches = list(range(0, N_AUDIT, 64))
with torch.inference_mode():
    for b_start in _tqdm(batches, desc="Pool d'audit", unit="batch", leave=True):
        xb    = audit_X[b_start:b_start + 64]
        batch = xb.shape[0]
        h_t   = torch.zeros(batch, HIDDEN_SIZE, device=DEVICE)
        c_t   = torch.zeros(batch, HIDDEN_SIZE, device=DEVICE)
        for t in range(W - 1):
            h_t, c_t = model.cell(xb[:, t, :], h_t, c_t)
        concat_list.append(torch.cat([h_t, xb[:, W - 1, :]], dim=1).cpu())

x_pool_audit = torch.cat(concat_list, dim=0).to(DEVICE)
print(f'Pool d audit : {x_pool_audit.shape}  '
      f'(attendu : [{N_AUDIT}, {HIDDEN_SIZE + INPUT_SIZE}])')

In [ ]:
# Élagage + régression symbolique sur les 4 portes T-KAN (section 4.4.7)
# x_pool_audit : (N, hidden_size + input_size)  entrées réelles des portes
report = extract_full_model_report(
    model,
    x_pool        = x_pool_audit,
    feature_names = FEATURE_COLS,
    theta         = 1e-2,
    r2_threshold  = 0.99,
)

# Statistiques globales
n_active   = sum(len(edges) for edges in report.values())
n_symbolic = sum(1 for edges in report.values()
                 for e in edges if e['symbolifiable'])

print('=== Rapport d audit COBAC ===')
print(f'Arêtes actives   (|φ|₁ > θ=0.01)  : {n_active}')
print(f'Arêtes symbolifiables (R² ≥ 0.99) : {n_symbolic} / {n_active}')
print()

for gate_name, edges in report.items():
    if not edges:
        print(f'Porte {gate_name.upper():10s}  aucune arête active')
        continue
    print(f'Porte {gate_name.upper():10s} ({len(edges)} arêtes actives) :')
    for edge in edges[:5]:   # top 5 par importance L1
        symb = 'OK ' if edge['symbolifiable'] else '~  '
        print(f"  [{symb}] {edge['input']:15s} -> {edge['output']:12s}  "
              f"L1={edge['l1_importance']:.4f}  "
              f"R2={edge['r2']:.4f}  "
              f"{edge['formula']}")
    print()

In [ ]:
import json

torch.save({
    'model_state': model.state_dict(),
    'config': {
        'input_size':   INPUT_SIZE,
        'hidden_size':  HIDDEN_SIZE,
        'W':            W,
        'M':            M_GAUSS,
        'K':            K_FOURIER,
        'feature_cols': FEATURE_COLS,
        'mu_train':     mu_train.to_dict(),
        'std_train':    std_train.to_dict(),
    },
    'test_metrics':  test_metrics,
    'best_val_mcc':  best_val_mcc,
}, os.path.join(CHECKPOINT_DIR, 'mkan_final.pt'))

with open(os.path.join(CHECKPOINT_DIR, 'audit_report.json'), 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)

print('Sauvegardé :')
print(f'  {CHECKPOINT_DIR}/mkan_final.pt          modèle + config + métriques')
print(f'  {CHECKPOINT_DIR}/audit_report.json       rapport d audit COBAC')
print(f'  {CHECKPOINT_DIR}/training_curves.html    courbes Plotly interactives')
print(f'  {CHECKPOINT_DIR}/confusion_matrix.html   matrice de confusion Plotly')

## 11. Visualisations structurelles des aretes T-KAN (MKANVisualizer)

`viz` a ete instancie a la section 7. On utilise ici `x_pool_audit` (section 10) pour les methodes necessitant les entrees reelles des portes.

- `plot_edge_heatmap` — heatmap L1 par porte (eq. 2.19)
- `plot_edge_functions` — courbes phi_ij(x) avec decomposition Gaussienne + Fourier (section 4.2.2)
- `plot_pruning_summary` — bilan de sparsification par porte (eq. 4.28)

In [ ]:
# Courbes de perte decomposees et metriques individuelles
for title, method in [
    ('loss_curves',     viz.plot_loss_curves),
    ('regularization',  viz.plot_regularization_detail),
    ('val_metrics',     viz.plot_metrics),
]:
    fig = method()
    fig.write_html(os.path.join(CHECKPOINT_DIR, f'{title}.html'))
    fig.show()


In [ ]:
from tqdm.auto import tqdm as _tqdm

for gate_name in _tqdm(['forget', 'input', 'candidate', 'output'],
                        desc="Heatmaps", unit="porte"):
    fig_hm = viz.plot_edge_heatmap(gate_name, x_pool_audit)
    fig_hm.write_html(os.path.join(CHECKPOINT_DIR, f'heatmap_{gate_name}.html'))
    fig_hm.show()
    _tqdm.write(f"  Heatmap {gate_name} sauvegardée.")

In [ ]:
from tqdm.auto import tqdm as _tqdm

for gate_name in _tqdm(['forget', 'input', 'candidate', 'output'],
                        desc="Edge functions", unit="porte"):
    try:
        fig_fn = viz.plot_edge_functions(gate_name, x_pool_audit, theta=1e-2, top_k=6)
        fig_fn.write_html(os.path.join(CHECKPOINT_DIR, f'edge_functions_{gate_name}.html'))
        fig_fn.show()
    except ValueError as e:
        _tqdm.write(f"  Porte {gate_name} ignorée : {e}")

In [ ]:
# Bilan de sparsification : arêtes totales vs survivantes par porte (eq. 4.28)
fig_prune = viz.plot_pruning_summary(x_pool_audit, theta=1e-2)
fig_prune.write_html(os.path.join(CHECKPOINT_DIR, 'pruning_summary.html'))
fig_prune.show()
print(f"Bilan élagage sauvegardé : {CHECKPOINT_DIR}/pruning_summary.html")
